# FINAL PROJECT

### Reconstruct 3D from stereoscopic side-by-side images

Student = David Medina Rosner

Id = F11115117

In [1]:
# libraries
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# > mental note
#  - opencv works with BGR
#    B(0) ~ R(2)
#  - matplotlib works with RGB

In [2]:
# left camera parameters
left_K = np.array([[1000.0, 0.0,    360.0],
                   [0.0,    1000.0, 640.0],
                   [0.0,    0.0,    1.0]])

left_RT = np.array([[0.88649035,   -0.46274707, -0.00,       -14.42428],
                    [-0.070794605, -0.13562201, -0.98822814, 86.532959],
                    [0.45729965,   0.8760547,   -0.1529876,  235.35446]])

left_P = left_K @ left_RT
left_P

array([[ 1.05111822e+03, -1.47367378e+02, -5.50755360e+01,
         7.03033256e+04],
       [ 2.21877171e+02,  4.25052998e+02, -1.08614020e+03,
         2.37159813e+05],
       [ 4.57299650e-01,  8.76054700e-01, -1.52987600e-01,
         2.35354460e+02]])

In [3]:
# right camera parameters
right_K = np.array([[1100.0, 0.0,    360.0],
                    [0.0,    1100.0, 640.0],
                    [0.0,    0.0,    1.0]])

right_RT = np.array([[0.98480779,   -0.17364818, -4.9342116E-8, -0.98420829],
                     [-0.026566068, -0.15066338, -0.98822814,   85.070221],
                     [0.17160401,   0.97321475,  -0.1529876,    236.97873]])

right_P = right_K @ right_RT
right_P

array([[ 1.14506601e+03,  1.59344312e+02, -5.50755903e+01,
         8.42297137e+04],
       [ 8.06038916e+01,  4.57127722e+02, -1.18496302e+03,
         2.45243630e+05],
       [ 1.71604010e-01,  9.73214750e-01, -1.52987600e-01,
         2.36978730e+02]])

In [4]:
# fundamental matrix
F = np.array([[3.283965767647195E-7,  -6.76098398189792E-6,  0.0021123144539793737],
              [-8.046341661808292E-6, 3.05632173594769E-8,   0.05124913199417346],
              [0.0048160232373805345, -0.051062699158041805, 1.0706286680443888]])

In [5]:
##################################    FUNCTIONS    #####################################################################################################

In [6]:
# function to open all images given
def open_all_images(file_path):
    img = cv.imread(file_path) # open image
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    
    return img

In [7]:
# display two kinds of images side-by-side
def display_images_side_by_side(image_1, title_1, image_2, title_2, start=0, end=179):
    for x in range(start, end):
        plt.figure(figsize=(15, 10))
        print('Image:', x)
        
        plt.subplot(1, 2, 1)
        plt.title(title_1)
        plt.imshow(image_1[x])
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.title(title_2)
        plt.imshow(image_2[x])
        plt.axis('off')
        
        plt.show()

In [8]:
# create a mask by using image segmentation on rectangle_1 and rectangle_2 for the given image, 
# return a mask image
def create_mask(image, rectangle_1, rectangle_2):
    
    # models for background and foreground
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    
    # apply GrabCut algorithm to rectangle_1
    mask = np.zeros(image.shape[:2], np.uint8)
    cv.grabCut(image, mask, rectangle_1, bgd_model, fgd_model, 5, cv.GC_INIT_WITH_RECT)
    
    # create binary mask for rectangle_1
    mask_1 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
    
    # apply GrabCut algorithm to rectangle_2
    mask = np.zeros(image.shape[:2], np.uint8)  # reset mask
    cv.grabCut(image, mask, rectangle_2, bgd_model, fgd_model, 5, cv.GC_INIT_WITH_RECT)
    
    # create binary mask for rectangle_2
    mask_2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
    
    # combine both masks
    combined_mask = np.maximum(mask_1, mask_2)
    
    # segment the image
    segmented_img = combined_mask * 255
    mask_rgb = cv.merge([segmented_img, segmented_img, segmented_img])

    return mask_rgb

In [9]:
# function to create binary image from blue pixels in picture
def blue_px_binary_picture(img, threshold=20):
    # convert image to float32 for precise calculation
    img_float = img.astype(np.float32)
    
    # calculate the differences
    # RGB format -> R(0) ~ B(2)
    diff_b_r = img_float[:,:,2] - img_float[:,:,0]
    diff_b_g = img_float[:,:,2] - img_float[:,:,1]
    
    # detect pixels where blue channel is greater than red and green channels by a threshold
    blue_px = (diff_b_r > threshold) & (diff_b_g > threshold)
    blue_px = blue_px.astype(np.uint8) * 255
    
    return blue_px

In [10]:
# split list of images into a list of left images, and a list of right images
def split_img(img_list):
    img_left = []
    img_right = []

    image_width = img_list[0].shape[1]
    
    for img in img_list:
        img_left.append(img[:, :image_width // 2])
        img_right.append(img[:, image_width // 2:])

    return img_left, img_right

In [11]:
# function to choose the first occurrence of a max value as single pixel per row
def create_single_line(binary_image):
    filtered_image = np.zeros_like(binary_image)
    
    # iterate over the rows of the image
    for idx, row in enumerate(binary_image):
        non_zero_indices = np.where(row > 0)[0]  # find indices of all non-zero values
        
        if non_zero_indices.size > 0:
            # find the first maximum occurrence to the right
            max_value = np.max(row)
            max_indices = np.where(row == max_value)[0]
            first_max_index = max_indices[-1]  # last occurrence to the right
            
            filtered_image[idx, first_max_index] = 255  # set the value at the first max index to 255
    
    return filtered_image

In [12]:
# get the xy coordinates from the binary images
def get_xy_coordinates_from_line(binary_image):
    points_list = []

    img_height = binary_image.shape[0]
    img_width = binary_image.shape[1]

    # get the first rightmost pixel as point
    for y in range(img_height):
        for x in range(img_width):
            if binary_image[y, x]:
                points_list.append([x, y, 1.0])
                break
                
    return np.array(points_list)

In [13]:
# given the a points list and the fundamental matrix, estimate the points in the binary
# image throught the epipolar line
def estimate_other_points(binary_img, points_list, F):
    estimated_points = []

    img_height = binary_img.shape[0]
    img_width = binary_img.shape[1]

    # get all epipolar lines for the given points
    for point in points_list:  
        epipolar_line = F @ point
        x, y, z = 0, 0, 0

        for x in range(img_width):
            if epipolar_line[1] != 0:
                y = int((-epipolar_line[0] * x - epipolar_line[2]) / epipolar_line[1])
            else:
                continue

            # if y is within the range of the image, and the value is non-zero
            if 0 <= y < img_height and binary_img[y, x]:
                z = 1
                break
        estimated_points.append([x, y, z])
        
    return np.array(estimated_points)

In [14]:
# perform 3D direct triangulation given two different point lists, and their respective
# P matrix
def direct_triangulation(points_L, points_R, P_L, P_R):
    # left
    P_L1 = P_L[0]
    P_L2 = P_L[1]
    P_L3 = P_L[2]

    # right
    P_R1 = P_R[0]
    P_R2 = P_R[1]
    P_R3 = P_R[2]

    triangulated_point = []

    # iterate every each point
    for (pl, pr) in zip(points_L, points_R):
        ul, vl, _ = pl
        ur, vr, _ = pr

        A = np.array([ul * P_L3 - P_L1,
                      vl * P_L3 - P_L2,
                      ur * P_R3 - P_R1,
                      vr * P_R3 - P_R2])

        # solved by SVD
        u, s, vT = np.linalg.svd(A)
        X = vT[3] / vT[3,3] # normalized

        # discard points with large error
        epsilon = A @ X
        error = np.linalg.norm(epsilon)

        # any further than 150 causes projection-errors
        if error <= 150:
            triangulated_point.append(X)
            
    return triangulated_point

In [15]:
# get the colors from both points, and compute the average to return a list of RGB colors
def add_color(image, points_1, points_2):

    color_list = []
    for (pt_1, pt_2) in zip(points_1, points_2):
        x_1, y_1, _ = pt_1
        x_2, y_2, _ = pt_2

        x_1, y_1 = int(x_1), int(y_1)
        x_2, y_2 = int(x_2), int(y_2)

        color_1 = image[y_1, x_1]
        color_2 = image[y_2, 719 + x_2]

        # compute the average color
        average_color = (color_1.astype(np.float32) + color_2.astype(np.float32)) / 2
        average_color = average_color.astype(np.uint8)

        color_list.append(average_color)
            
    return color_list

In [16]:
##################################    CODE  IMPLEMENTATION   ############################################################################################

In [17]:
# file_paths for all images in the folder
file_paths = ['SBS images/{:03d}.jpg'.format(x) for x in range(179)]

# list of original images
images_list = [open_all_images(img) for img in file_paths]

In [18]:
# create a mask based on image '000.jpg'
left_roi = (150, 55, 347, 814) 
right_roi = (927, 7, 350, 881)

mask = create_mask(images_list[0], left_roi, right_roi)

In [19]:
# apply mask to all images
filtered_image_list = [cv.bitwise_and(img, mask) for img in images_list]

In [20]:
# list of binary images for BLUE LINE
binary_image_list = [blue_px_binary_picture(img) for img in filtered_image_list]

In [21]:
# split image into left and right ones
binary_left, binary_right = split_img(binary_image_list)

In [22]:
# create single pixel line for binary left images
single_px_left_list = [create_single_line(img) for img in binary_left]

In [23]:
single_px_right_list = [create_single_line(img) for img in binary_right]

In [24]:
# use LEFT points to project to RIGHT points

# get the xy coordinates from binary left image
points_left = [get_xy_coordinates_from_line(img) for img in single_px_left_list]

# estimate the right points throught epipolar line
points_right = [estimate_other_points(br, pl, F) for (br, pl) in zip(binary_right, points_left)]

# form left points and estimated right points, perform triangulation to get 3D points
points_3d = [direct_triangulation(pl, pr, left_P, right_P) for (pl, pr) in zip(points_left, points_right)]

In [25]:
# create a color list for each point
color_list = [add_color(images_list[0], p_l, p_r) for (p_l, p_r) in zip(points_left, points_right)]

In [26]:
# use RIGHT points to project to LEFT points

# get the xy coordinates from binary left image
points_right_ = [get_xy_coordinates_from_line(img) for img in single_px_right_list]

# estimate the right points throught epipolar line
points_left_ = [estimate_other_points(bl, pr, F.T) for (bl, pr) in zip(binary_left, points_right_)]

# form left points and estimated right points, perform triangulation to get 3D points
points_3d_ = [direct_triangulation(pr, pl, right_P, left_P) for (pr, pl) in zip(points_right_, points_left_)]

In [27]:
with open('F11115117.xyz', 'w') as fp:
    nro = 0
    # save triangulated points from LEFT to RIGHT
    for (img_colors, img_points) in zip(color_list, points_3d):
        for (rgb, pt) in zip(img_colors, img_points):
            x, y, z, w = pt
            r, g, b = rgb
            
            if abs(x) > 200 or abs(y) > 45 or abs(z) > 200:
                continue
            fp.write(f"{x:.4f} {y:.4f} {z:.4f} {r} {g} {b}\n")
            nro += 1

    # save triangulated points from RIGHT to LEFT
    for (img_colors, img_points) in zip(color_list, points_3d_):
        for (rgb, pt) in zip(img_colors, img_points):
            x, y, z, w = pt
            r, g, b = rgb
            
            if abs(x) > 200 or abs(y) > 45 or abs(z) > 200:
                continue
            fp.write(f"{x:.4f} {y:.4f} {z:.4f} {r} {g} {b}\n")
            nro += 1
            
    print(f'SuccessfuLly saved! {nro} points.')     

SuccessfuLly saved! 93656 points.


In [28]:
##################################    FINAL    ##########################################################################################################